In [7]:
import requests
import pandas as pd

BASE_URL = "https://gamma-api.polymarket.com/markets"

def fetch_markets(limit=500):
    params = {
        "limit": limit,
        "offset": 0,
        "closed": "true"
    }
    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()
    return response.json()

data = fetch_markets()
print(data[0].keys())
keys = list(data[0].keys())
for key in keys:
    print(key, data[0].get(key))
print(len(data))

dict_keys(['id', 'question', 'conditionId', 'slug', 'twitterCardImage', 'endDate', 'category', 'liquidity', 'image', 'icon', 'description', 'outcomes', 'outcomePrices', 'volume', 'active', 'marketType', 'closed', 'marketMakerAddress', 'updatedBy', 'createdAt', 'updatedAt', 'closedTime', 'mailchimpTag', 'archived', 'restricted', 'volumeNum', 'liquidityNum', 'endDateIso', 'hasReviewedDates', 'readyForCron', 'volume24hr', 'volume1wk', 'volume1mo', 'volume1yr', 'clobTokenIds', 'fpmmLive', 'volume1wkAmm', 'volume1moAmm', 'volume1yrAmm', 'volume1wkClob', 'volume1moClob', 'volume1yrClob', 'events', 'creator', 'ready', 'funded', 'cyom', 'competitive', 'pagerDutyNotificationEnabled', 'approved', 'rewardsMinSize', 'rewardsMaxSpread', 'spread', 'oneDayPriceChange', 'oneHourPriceChange', 'oneWeekPriceChange', 'oneMonthPriceChange', 'oneYearPriceChange', 'lastTradePrice', 'bestBid', 'bestAsk', 'clearBookOnStart', 'manualActivation', 'negRiskOther', 'umaResolutionStatuses', 'pendingDeployment', 'dep

In [26]:
import json

def is_binary_market(market):
    outcomes = market.get("outcomes", [])
    outcomes = json.loads(outcomes)
    return len(outcomes) == 2

def is_resolved(market):
    return market.get("closed") is True
    # outcomePricesStr = json.loads(market.get("outcomePrices"))
    # outcomePrices = [float(x) for x in outcomePricesStr]
    # return (outcomePrices == [0.0, 1.0] or outcomePrices == [1.0, 0.0])

def has_min_duration(market, min_days=30):
    try:
        start = pd.to_datetime(market.get("startDate"))
        end = pd.to_datetime(market.get("endDate"))
        return (end - start) >= pd.Timedelta(days=min_days)
    except:
        return False
    
def is_valid_market(market):
    return (
        is_binary_market(market) and
        is_resolved(market) and
        (market.get("volumeNum", 0) > 0) and
        has_min_duration(market)
    )

filtered = [m for m in data if is_valid_market(m)]

print("Filtered markets:", len(filtered))

Filtered markets: 1


In [9]:
BASE_URL = "https://data-api.polymarket.com"

def get_recent_trades(limit=20):
    url = f"{BASE_URL}/trades"
    params = {"limit": limit}

    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    return r.json()

def main():
    trades = get_recent_trades(limit=20)

    df = pd.DataFrame(trades)
    print("Columns:", df.columns.tolist())
    print()
    print(df.head())

if __name__ == "__main__":
    main()

Columns: ['proxyWallet', 'side', 'asset', 'conditionId', 'size', 'price', 'timestamp', 'title', 'slug', 'icon', 'eventSlug', 'outcome', 'outcomeIndex', 'name', 'pseudonym', 'bio', 'profileImage', 'profileImageOptimized', 'transactionHash']

                                  proxyWallet side  \
0  0xf1336f4007f01120f7020ba5882ae539340c66be  BUY   
1  0xc679408823e9c15c5099bc055c8c872d7f9e4ae5  BUY   
2  0x73839b90fa83a860a8b44f629127c432f7c24e68  BUY   
3  0x20d2309cd92b797ae7ca175ed828ed8a27fbe29d  BUY   
4  0x01b739b360d3c2f6cc8ec84cda900d48650e2eca  BUY   

                                               asset  \
0  4164307741742690126378616194523248679078470833...   
1  9786208035581705891492024366991853343883663155...   
2  2034068953857625408646103829425395404954351290...   
3  4541861876378788579466939506077202933830820603...   
4  4541861876378788579466939506077202933830820603...   

                                         conditionId       size     price  \
0  0x4045f5afecd0d6b

In [42]:
# def fetch_price_history(token_id):
#     url = f"{BASE_URL}/trades"

#     all_data = []
#     offset = 0
#     limit = 1000

#     while True:
#         params = {
#             "token": token_id,
#             "limit": limit,
#             "offset": offset
#         }

#         r = requests.get(url, params=params, timeout=10)
#         if r.status_code != 200:
#             print("oopsie")
        
#         chunk = r.json()
#         print(chunk)

#         if not chunk:
#             print("not chunk")
#             break

#         all_data.extend(chunk)

#         if len(chunk) < limit:
#             break

#         offset += limit

#     df = pd.DataFrame(all_data)

#     if df.empty:
#         return None

#     df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s")
#     df = df.sort_values("timestamp")

#     price_series = (
#         df.set_index("timestamp")["price"]
#         .resample("1h")
#         .last()
#         .ffill()
#         .reset_index()
#     )

#     return price_series

def fetch_price_history(token_id):
    url = f"{BASE_URL}/trades"

    all_data = []
    offset = 0
    limit = 1000
    MAX_OFFSET = 3000

    while True:
        if offset >= MAX_OFFSET:
            print("Reached API offset limit")
            break

        params = {
            "token": token_id,
            "limit": limit,
            "offset": offset
        }

        r = requests.get(url, params=params, timeout=10)

        if r.status_code != 200:
            print("Request failed:", r.status_code, r.text)
            break

        chunk = r.json()

        # 🔴 safety check: ensure it's actually a list of trades
        if not isinstance(chunk, list):
            print("Bad response format:", chunk)
            break

        if len(chunk) == 0:
            break

        all_data.extend(chunk)

        if len(chunk) < limit:
            break

        offset += limit

    if not all_data:
        return None

    df = pd.DataFrame(all_data)

    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s")
    df = df.sort_values("timestamp")

    price_series = (
        df.set_index("timestamp")["price"]
        .resample("1h")
        .last()
        .ffill()
        .reset_index()
    )

    return price_series

tokens = json.loads(filtered[0].get("clobTokenIds", "[]"))
print(tokens[0])
df = fetch_price_history(tokens[0])
print(df.columns)
print(df['price'].head())

df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s")

df = df.sort_values("timestamp")

print(df[['price', 'timestamp']])

price_series = (
    df.set_index("timestamp")["price"]
    .resample("1h")
    .last()
    .ffill()
)

print("price series", price_series)

22551521971148855406938873027996883656661184949232892000317323701135896166555
Reached API offset limit
Index(['timestamp', 'price'], dtype='str')
0    0.530817
Name: price, dtype: float64
      price           timestamp
0  0.530817 2026-04-27 05:00:00
price series timestamp
2026-04-27 05:00:00    0.530817
Freq: h, Name: price, dtype: float64


In [30]:
def get_price_at_horizon(price_df, resolution_time, days_before):
    target_time = resolution_time - pd.Timedelta(days=days_before)

    df_before = price_df[price_df["timestamp"] <= target_time]

    if df_before.empty:
        return None

    return df_before.iloc[-1]["price"]

def build_market_features(price_df, resolution_time):
    return {
        "p_30d": get_price_at_horizon(price_df, resolution_time, 30),
        "p_7d": get_price_at_horizon(price_df, resolution_time, 7),
        "p_1d": get_price_at_horizon(price_df, resolution_time, 1),
    }

def get_outcome(market):
    return int(market["outcome"] == "Yes")  # adjust if needed

In [ ]:
dataset = []

for market in filtered:
    try:
        tokens = json.loads(market["clobTokenIds"])
        token_id = tokens[0]

        price_df = fetch_price_history(token_id)
        if price_df is None:
            continue

        resolution_time = pd.to_datetime(market["endDate"])
        outcome = get_outcome(market)

        features = build_market_features(price_df, resolution_time)

        row = {
            "market_id": market["id"],
            "outcome": outcome,
            **features
        }

        dataset.append(row)

    except Exception as e:
        continue

df_final = pd.DataFrame(dataset)
print(df_final.head())

price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history
price history


KeyboardInterrupt: 